# Task4 — Class-specific probability stacking (CPU only)

**ไม่ฝึก CNN ใหม่ ไม่ใช้ GPU ไม่เปิด Test**. Logistic regression เรียนจาก centered log-probabilities 15ค่า ของ ResNetสองตัวและEfficientNetเดิม แล้วผสมprediction stacker50%+baseline50% น้ำหนักนี้ล็อกก่อนรัน

เหตุผล: fixed ensemble ให้น้ำหนักเดียวทุกคลาส แต่ตัวรวมแบบเรียนรู้สามารถให้น้ำหนักและแก้confusionต่างกันตามคลาสได้. ต้องระวัง overfit จึงใช้ regularization C=.1 และเว้นทีละfold

Train meta-modelจาก4folds ประเมินfoldที่5 โดยทุกcorruption versionของบัญชีเดียวกันอยู่foldเดียวกัน. ใช้ทั้งoriginalและsyntheticstress โดยให้massรวมอย่างละครึ่ง และแบ่งstressเท่ากันตามfamily. ไม่มีการค้นC/blendจากTest

**ข้อจำกัดสำคัญ:** เป็นcross-fold meta-evaluation ไม่ใช่nested CVเต็มรูปแบบ: base modelsของ4foldsที่ใช้ฝึกstackerเคยเห็นภาพของheld foldในtrainingของตัวเอง. OOFและsuiteถูกใช้ซ้ำ จึงมีโอกาสoptimistic. ผลผ่านยังไม่รับรองคะแนนKaggle

Run Allประเมินครบ5foldsจากcache, auditเฉพาะเมื่อผ่าน และfit stackerสุดท้ายถ้าผ่านทั้งหมด. ไม่exportหรือส่งKaggle. คาดว่าเบากว่าการฝึกCNNมาก แต่ยังไม่ได้วัดเวลาบนข้อมูลเต็ม

In [1]:
from pathlib import Path
import sys,json,time
import pandas as pd
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'Task4/models/image_training.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from Task4.tools import stacking_study as study
from Task4.models.image_audit import make_folds
from Task4.models.image_seed_evaluation import summarize
groups=pd.read_parquet(ROOT/'Task4/data/audit/followup_v2/train_groups.parquet')
folds=make_folds(groups,mode='near',n_splits=5,seed=42)
OUT=ROOT/'Task4/models/results/class_specific_stacking_v1'


## 1. Cross-fold evaluation และ audit
ใช้เฉพาะprediction cacheจากTrain; cacheไม่ครบจะหยุดแทนการเริ่มGPUinference

In [2]:
started=time.monotonic()
scores,result=study.run(ROOT,groups,folds,OUT)
print('Elapsed minutes:',round((time.monotonic()-started)/60,1))
display(summarize(scores))
display(result)

Meta fold 1 complete
Meta fold 2 complete
Meta fold 3 complete
Meta fold 4 complete
Meta fold 5 complete
Elapsed minutes: 0.0


suite,compound,original,single
procedure,,,
baseline,0.946368,0.983914,0.981367
stack_blend50,0.948866,0.983293,0.980797


{'original_gain': -0.000621516273126188,
 'stress_family_gain': -5.925333670365315e-05,
 'stress_wins': 2,
 'worst_class_gain': -0.0013207187518310848,
 'worst_condition_gain': -0.002830744518507644,
 'passed': False,
 'audit_evaluated': False,
 'test_used': False}

## 2. หลักฐานแยกfoldและcondition

In [3]:
table=scores.pivot(index=['fold','condition'],columns='procedure',values='macro_F1')
display(table.assign(gain=table.stack_blend50-table.baseline))
print('Saved results:',OUT)
print('No Test inference or Kaggle submission performed.')

procedure                       baseline  stack_blend50      gain
fold condition                                                   
1    blur_1                     0.982007       0.983978  0.001971
     blur_2                     0.976005       0.978005  0.002000
     compound_bright_color      0.947935       0.950008  0.002073
     compound_color_noise       0.949996       0.958061  0.008065
     compound_dark_color_noise  0.927962       0.939967  0.012005
...                                  ...            ...       ...
5    jpeg_1                     0.982027       0.980035 -0.001992
     jpeg_2                     0.979978       0.979969 -0.000009
     original                   0.984526       0.984965  0.000440
     speckle_1                  0.984027       0.982027 -0.001999
     speckle_2                  0.986018       0.975989 -0.010029

[85 rows x 3 columns]

Saved results: /Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/Task4/models/results/class_specific_stacking_v1
No Test inference or Kaggle submission performed.
